# Hierarchical pooling across an item's stores, on the M5 subset

Issue #156, part of the #143 re-plan (references in #144).

**Question.** The re-plan treats each item as a multivariate series over its 10 stores, but every fit so far has been a
no-pooling fit: each store has its own coefficients, its own weekly pattern and its own level. Per-series coefficients
as filter states (stage 1, #146) lost on the full run, and `SspReg` with a per-series `beta` does not do well either.
Does sharing something across an item's 10 stores help: the covariate coefficients, the weekly pattern, the level
noise? Which one or two variants are worth registering as M5 entries?

**Method.** Everything is scored on the 3-item × 10-store benchmark subset of #154 (30 series, `m5_subset.py`) at the
pre-validation origin, by the 12-level WRMSSE of `m5_score_submission.py`, with the fit time beside it, as in `ssp_12`.
No full run is made. All variants share the #118 default's Yeo-Johnson target (fitted once per series and held fixed)
and its half-normal sigma priors. The variants that have covariates use the 73 `CALENDAR_COVARIATES` columns as a static
regression `X beta` outside the filter, as `SspReg` does:

| group | variant | what is pooled | how it is fitted |
|---|---|---|---|
| baseline | #118 default | nothing, and no covariates | MAP (Adam) |
| no model change | bottom-up, OLS, WLS, MinT | the forecasts, after the fit | the default's forecasts of the 10 stores and of the item total, reconciled |
| coefficients | no, full and partial pooling of `beta` | `beta` = item part + store deviation, with prior variance `tau^2` | MAP of the independent filters (`tau` chosen on a holdout), then NUTS for one `tau` |
| weekly pattern | none, full and partial pooling of a static weekday pattern | the 6 weekday offsets, as static coefficients | MAP of a joint filter of the item's 10 stores |
| weekly pattern | one drifting weekday pattern shared by the 10 stores, plus a store deviation | the drifting offsets | MAP of the joint filter |
| store levels | independent, equicorrelated, one-factor and full-covariance level noise (SUTSE) | the level innovations | MAP of the joint filter |

"Pooling" is a prior on the coefficients of one item's stores: `beta_s = b + delta_s`, `b ~ N(0, 1)` shared by the item's
stores and `delta_s ~ N(0, tau^2)` the store's deviation. `tau = 0` is full pooling and a large `tau` is no pooling; both
pieces are static, so the model stays linear Gaussian. Pooling across items is not tried: three items from three
categories cannot support it.

There are two ways to build pooling, and the notebook tries both:

* **Coefficients as parameters** (`ssp_reg_model`): pooling changes only the prior of `beta`, and each store's filter
  stays independent given `beta` (the diagonal filter of `kalman_filter_1d_batch`).
* **A joint filter per item**, 10 observations a day with the covariance between the states kept. The weekly and level
  variants need it, since they share filter states. `bunobee` has no such filter, so a small one is written here
  (`joint_filter`) and checked against the batch filter.

**Runtime.** On a laptop CPU the MAP fits take 2 s (independent filters) to 20 s (joint filter), the one NUTS fit takes
about 6 minutes, and the notebook runs in about 21 minutes end to end. The long cells besides NUTS are the choice of the
weekly `tau` on three rolling folds (4 minutes) and the check at the other two origins.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import jax
import jax.numpy as jnp
import jax.scipy as jsp
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import numpyro.distributions as dist
import optax
import pandas as pd
import xarray as xr
from numpyro.diagnostics import effective_sample_size
from numpyro.infer import MCMC, NUTS

warnings.filterwarnings("ignore")

# The M5 scripts live in playground/m5_prototype and import each other by name.
m5_dir = next(p / "m5_prototype" for p in [Path.cwd(), *Path.cwd().parents] if (p / "m5_prototype").is_dir())
sys.path.insert(0, str(m5_dir))

from m5_common import load_m5_ds  # noqa: E402
from m5_models import CALENDAR_COVARIATES, DEFAULT_MODEL, HALF_NORMAL_PRIORS, MODELS  # noqa: E402
from m5_score_submission import compute_hierarchical_wrmsse, window_masks  # noqa: E402
from m5_subset import SUBSET_ITEMS, per_item_wrmsse, subset_series_idx  # noqa: E402

from bunobee.models.m5.m5_ssp_optim import _softplus, _softplus_inv, kalman_filter_1d_batch  # noqa: E402
from bunobee.models.m5.m5_ssp_reg import predict_batch_series_reg  # noqa: E402
from bunobee.models.m5.transforms import PowerTransform  # noqa: E402
from bunobee.regression import make_periodic_dummies  # noqa: E402

print("numpyro", numpyro.__version__, "| jax", jax.__version__, "| devices", jax.devices())

## Data, and the score

`score` is the pre-validation WRMSSE of the subset: the mean over the 12 levels, as `m5_run.py` reports it, plus the
score of each item alone (`high` is `FOODS_3_586`, `mid` `HOUSEHOLD_1_069`, `low` `HOBBIES_1_197`), since the high item
holds most of the dollar sales. The subset's series come item by item, 10 stores each, in the same store order, which the
pooled fits rely on.

In [ ]:
ds = load_m5_ds()
series_idx = subset_series_idx(ds)  # 3 items x 10 stores, item by item (high, mid, low)
fit_mask, forecast_mask = window_masks(ds["split"].values, "prevalidation")
base_ids = ds["series_id"].values[series_idx]
sales = ds["sales"].values[series_idx][:, fit_mask]  # (30, n_fit_days)
n_series, n_fit = sales.shape
N_ITEMS, N_STORES = len(SUBSET_ITEMS), n_series // len(SUBSET_ITEMS)
stores = ds["store_id"].values[series_idx].reshape(N_ITEMS, N_STORES)
assert (stores == stores[0]).all(), "every item must list its stores in the same order"
print(f"{n_series} series, {n_fit} fit days, {forecast_mask.sum()} forecast days;  items: {SUBSET_ITEMS}")
print("stores:", list(stores[0]))

# The design: the #118 states (a level and six weekday offsets) and the stage-1 calendar covariates.
weekday = ds["day_of_week"].values
Z_all = make_periodic_dummies(weekday.size, period=7, phase=int(weekday[0]), intercept=True)  # (n_dates, 7)
X_all = CALENDAR_COVARIATES.design(ds)  # (n_dates, K)
Z, X = Z_all[fit_mask], X_all[fit_mask]
Z_future, X_future = Z_all[forecast_mask], X_all[forecast_mask]
K = X.shape[1]
transform = PowerTransform("yeo-johnson")  # the transform of the #118 default, fitted once per series
SIGMA_H, SIGMA_Q = HALF_NORMAL_PRIORS["sigma_h_prior"], HALF_NORMAL_PRIORS["sigma_q_prior"]
print(f"Z {Z.shape}, X {X.shape}")


def score(forecasts, window="prevalidation"):
    """WRMSSE: the mean over the 12 levels of the subset's forecasts, and the score of each item alone."""
    f = np.asarray(forecasts, dtype=np.float64)
    overall = compute_hierarchical_wrmsse(ds, f, base_ids, window=window)["wrmsse"].mean()
    items = per_item_wrmsse(ds, f, base_ids, window=window)
    return float(overall), items


RESULTS = []


def record(name, forecasts, seconds, group="", **extra):
    """Score a forecast, keep a row for the final table and return it."""
    overall, items = score(forecasts)
    per_item = {f"wrmsse_{tier}": v for tier, v in zip(items["tier"], items["wrmsse"])}
    row = {"approach": name, "group": group, "wrmsse": overall, **per_item, "seconds": seconds, **extra}
    RESULTS.append(row)
    return row

## The score to beat

The #118 default: each store on its own, MAP, a level and six drifting weekday offsets, no covariates. Its forecasts
are also the base forecasts of the reconciliation baseline below.

In [ ]:
t0 = time.time()
f_default, _ = MODELS[DEFAULT_MODEL].fit_forecast(ds, series_idx, fit_mask, forecast_mask)
t_default = time.time() - t0
print(record("#118 default (MAP, no covariates)", f_default, t_default, group="baseline"))

## No model change: reconcile independent forecasts so they add up

The item's 10 stores sum to the item total. Bottom-up is the default's 10 forecasts, summed by the scorer, so its score is
the default's. Optimal combination also uses an independent forecast of the item total (the default fitted on the 3 item
series) and adjusts all 11 forecasts to add up: `y_tilde = S (S' W^-1 S)^-1 S' W^-1 y_hat`, with `S` the 11 × 10
summing matrix. `W` is the identity (OLS), the structural `diag(S 1)` (WLS), or the covariance of the base forecast errors
shrunk toward its diagonal (MinT-shrink). The errors are the 28-day errors of three rolling origins at the end of the fit
window, so they never see the window being scored. The reconciled forecasts are cut at 0.

In [ ]:
S = np.vstack([np.ones((1, N_STORES)), np.eye(N_STORES)])  # (11, 10): the item total, then the 10 stores
totals = ds["sales"].values[series_idx].reshape(N_ITEMS, N_STORES, -1).sum(axis=1)  # (3, n_dates)
ds_total = xr.Dataset(  # the same dataset, with the 3 item totals as the series, for the default model
    {"sales": (("series_id", "date"), totals), "day_of_week": ds["day_of_week"]},
    coords={"series_id": [f"total{i}" for i in range(N_ITEMS)], "date": ds["date"]},
)
fit_days = np.flatnonzero(fit_mask)
HOLDOUT = 28  # days per fold
CUTS = [n_fit - k * HOLDOUT for k in (3, 2, 1)]  # three rolling origins at the end of the fit window: no look-ahead


def default_forecast(dset, idx, cut=None):
    """The #118 default fitted on the first ``cut`` fit days (all of them if None), with the next 28 days' actuals."""
    fm = np.zeros_like(fit_mask)
    fm[fit_days[: cut or n_fit]] = True
    if cut is None:
        fc = forecast_mask
    else:
        fc = np.zeros_like(fit_mask)
        fc[fit_days[cut : cut + HOLDOUT]] = True
    f, _ = MODELS[DEFAULT_MODEL].fit_forecast(dset, idx, fm, fc)
    return f, dset["sales"].values[idx][:, fc]


def shrunk_covariance(E):
    """Schafer-Strimmer shrinkage of the correlations of the rows of ``E`` toward 0; returns it with the intensity."""
    n, N = E.shape
    cov = E @ E.T / N
    sd = np.sqrt(np.diag(cov))
    Es = E / sd[:, None]
    corr = Es @ Es.T / N
    w = Es[:, None, :] * Es[None, :, :]
    var_r = N / (N - 1) ** 3 * ((w - w.mean(axis=2)[:, :, None]) ** 2).sum(axis=2)
    off = ~np.eye(n, dtype=bool)
    lam = float(np.clip(var_r[off].sum() / (corr[off] ** 2).sum(), 0, 1))
    shrunk = (1 - lam) * corr
    np.fill_diagonal(shrunk, 1.0)
    return shrunk * np.outer(sd, sd), lam


def combine(f_stores, f_totals, covariances):
    """Optimal combination of the stores' and item totals' forecasts, item by item, cut at 0."""
    out = np.zeros_like(f_stores)
    for i, W in enumerate(covariances):
        Wi = np.linalg.inv(W)
        G = np.linalg.solve(S.T @ Wi @ S, S.T @ Wi)  # (10, 11): the reconciled stores from the 11 base forecasts
        rows = slice(i * N_STORES, (i + 1) * N_STORES)
        out[rows] = G @ np.vstack([f_totals[i : i + 1], f_stores[rows]])
    return np.maximum(out, 0.0)


t0 = time.time()
f_total, _ = default_forecast(ds_total, np.arange(N_ITEMS))
t_base = t_default + time.time() - t0  # the cost of the base forecasts: the stores' fit and the totals' fit
errors = []  # per fold: (item, 11, 28) errors of the item total and of its 10 stores
for cut in CUTS:
    f_s, y_s = default_forecast(ds, series_idx, cut)
    f_t, y_t = default_forecast(ds_total, np.arange(N_ITEMS), cut)
    errors.append(np.concatenate([(y_t - f_t)[:, None, :], (y_s - f_s).reshape(N_ITEMS, N_STORES, -1)], axis=1))
errors = np.concatenate(errors, axis=2)  # (3, 11, 84)
shrunk = [shrunk_covariance(errors[i]) for i in range(N_ITEMS)]
print("MinT shrinkage intensity by item:", [round(lam, 3) for _, lam in shrunk])

for label, covs in (
    ("OLS", [np.eye(11)] * N_ITEMS),
    ("WLS, structural", [np.diag(S.sum(axis=1))] * N_ITEMS),
    ("MinT-shrink", [W for W, _ in shrunk]),
):
    t0 = time.time()
    f = combine(f_default, f_total, covs)
    print(record(f"Reconciled, {label}", f, t_base + time.time() - t0, group="no model change"))
print(
    "item total forecast / sum of the stores' forecasts, by item:",
    (f_total / f_default.reshape(3, 10, -1).sum(1)).mean(1).round(3),
)

## Pooling the covariate coefficients: only the prior of `beta` changes

The independent-filter model of `ssp_reg_model`: a level and six drifting weekday offsets per store, the covariates as a
static `X beta` outside the filter. `beta[item, store] = b[item] + delta[item, store]` with `b ~ N(0, tau_item^2)` and
`delta ~ N(0, tau_store^2)`:

* **no pooling:** no `b`, `delta ~ N(0, tau_store^2)`. With `tau_store` = 1 this is the ridge MAP of `ssp_12` (0.5227 there).
* **full pooling:** `b` only, the same 73 coefficients for the item's 10 stores.
* **partial pooling:** both, `tau_item` = 1 and `tau_store` in a grid.

The fit is MAP, a point estimate of the coefficients and sigmas by Adam, as in `ssp_12`, where it matched or beat the
NUTS fits at 1/100 of the cost. `tau_store` is chosen on the mean RMSSE of the 28-day forecasts of the three rolling folds
at the end of the fit window, so it never sees the origin being scored.

In [ ]:
def fit_pooled_map(sales, Z, X, tau_item=None, tau_store=None, n_iter=300, lr=3e-2):
    """MAP fit of independent filters whose covariate coefficients are pooled across each item's stores.

    ``beta = b[item] + delta[store]`` with ``b ~ N(0, tau_item^2)`` and ``delta ~ N(0, tau_store^2)``; a ``None`` scale
    leaves that part out (``tau_item=None``: no pooling, ``tau_store=None``: full pooling). The loss is the Kalman
    filter's log-likelihood of ``y - X beta`` plus the sigma and coefficient priors. Returns a dict shaped like
    ``fit_batch_series_reg``'s, with one "draw", so that ``predict_batch_series_reg`` forecasts from it.
    """
    params_t = transform.fit(sales)
    y = jnp.asarray(transform.forward(sales, params_t))
    B, n_states = y.shape[0], Z.shape[1]
    Zj, Xj = jnp.asarray(Z, dtype=jnp.float32), jnp.asarray(X, dtype=jnp.float32)

    def make_beta(p):
        beta = jnp.zeros((B, Xj.shape[1]))
        if tau_item is not None:
            beta = beta + jnp.repeat(p["b"], B // N_ITEMS, axis=0)
        if tau_store is not None:
            beta = beta + p["delta"]
        return beta

    def neg_log_post(p):
        sigma_h, sq_level, sq_dow = (_softplus(p["raw"][:, i]) for i in range(3))
        sigma_q = jnp.column_stack([sq_level, jnp.broadcast_to(sq_dow[:, None], (B, n_states - 1))])
        lp, *_ = kalman_filter_1d_batch(
            a0=jnp.zeros(n_states),
            P0=jnp.ones(n_states),
            Z=Zj,
            sigma_h=sigma_h,
            sigma_q=sigma_q,
            y=y - make_beta(p) @ Xj.T,
            logp=True,
            chunk_size=None,
        )
        total = (
            lp
            + SIGMA_H.distribution.log_prob(sigma_h)
            + SIGMA_Q.distribution.log_prob(sq_level)
            + SIGMA_Q.distribution.log_prob(sq_dow)
        ).sum()
        if tau_item is not None:
            total += dist.Normal(0.0, tau_item).log_prob(p["b"]).sum()
        if tau_store is not None:
            total += dist.Normal(0.0, tau_store).log_prob(p["delta"]).sum()
        return -total

    p = {"raw": jnp.tile(jnp.array([_softplus_inv(SIGMA_H.init), *[_softplus_inv(SIGMA_Q.init)] * 2]), (B, 1))}
    if tau_item is not None:
        p["b"] = jnp.zeros((N_ITEMS, Xj.shape[1]))
    if tau_store is not None:
        p["delta"] = jnp.zeros((B, Xj.shape[1]))
    opt = optax.adam(lr)
    state = opt.init(p)

    @jax.jit
    def step(p, state):
        loss, grads = jax.value_and_grad(neg_log_post)(p)
        updates, state = opt.update(grads, state)
        return optax.apply_updates(p, updates), state, loss

    for _ in range(n_iter):
        p, state, loss = step(p, state)
    sigma_h, sq_level, sq_dow = (np.asarray(_softplus(p["raw"][:, i]))[None] for i in range(3))
    return {
        "samples": {
            "sigma_h": sigma_h,
            "sigma_q_level": sq_level,
            "sigma_q_dow": sq_dow,
            "beta": np.asarray(make_beta(p))[None],
        },
        "y": np.asarray(y),
        "transform_params": params_t,
        "transform": transform,
        "Z": Zj,
        "X": np.asarray(Xj),
    }


def holdout_rmsse(forecast, actual, train):
    """RMSSE of a holdout forecast per series, scaled by the in-sample one-step naive error of the training days."""
    scale = np.maximum(np.mean(np.diff(train, axis=1) ** 2, axis=1), 1e-8)
    return np.sqrt(np.mean((forecast - actual) ** 2, axis=1) / scale)


def choose(fit_forecast, grid):
    """The grid value of the lowest holdout RMSSE over the three rolling folds; ``fit_forecast(sales, cut, value)``."""
    out = {}
    for value in grid:
        errs = []
        for cut in CUTS:
            pred = fit_forecast(sales[:, :cut], cut, value)
            errs.append(np.mean(holdout_rmsse(pred, sales[:, cut : cut + HOLDOUT], sales[:, :cut])))
        out[value] = float(np.mean(errs))
    return min(out, key=out.get), out


def forecast_pooled(sales_, cut, tau_store, **kw):
    res = fit_pooled_map(sales_, Z[:cut], X[:cut], **kw, tau_store=tau_store)
    return predict_batch_series_reg(res, Z[cut : cut + HOLDOUT], X[cut : cut + HOLDOUT], n_draws=1)

In [ ]:
TAUS = (0.03, 0.1, 0.3, 1.0)  # the store deviation's scale; tau = 0 is full pooling and a large tau is no pooling

# No pooling: the store's own coefficients under N(0, 1), the ridge MAP of ssp_12.
t0 = time.time()
res = fit_pooled_map(sales, Z, X, tau_store=1.0)
f_nopool = predict_batch_series_reg(res, Z_future, X_future, n_draws=1)
print(record("beta: no pooling (ridge, tau=1)", f_nopool, time.time() - t0, group="coefficients"))

# Full pooling: one set of 73 coefficients per item.
t0 = time.time()
res = fit_pooled_map(sales, Z, X, tau_item=1.0)
f_fullpool = predict_batch_series_reg(res, Z_future, X_future, n_draws=1)
print(record("beta: full pooling (item only)", f_fullpool, time.time() - t0, group="coefficients"))

# Partial pooling: the item part and a store deviation, with tau chosen on the rolling folds.
tau_best, tau_holdout = choose(lambda s, cut, tau: forecast_pooled(s, cut, tau, tau_item=1.0), TAUS)
print("holdout RMSSE by tau:", {k: round(v, 4) for k, v in tau_holdout.items()}, "-> tau =", tau_best)
t0 = time.time()
res = fit_pooled_map(sales, Z, X, tau_item=1.0, tau_store=tau_best)
f_partial = predict_batch_series_reg(res, Z_future, X_future, n_draws=1)
print(
    record(
        f"beta: partial pooling (tau={tau_best:g}, holdout choice)", f_partial, time.time() - t0, group="coefficients"
    )
)

In [ ]:
# Sensitivity of the pre-validation score to tau. This reads the score of the origin being scored, so it only shows how
# flat the choice is: the holdout row above is the one to compare. tau -> 0 is full pooling, tau large is no pooling.
grid = []
for tau in (0.01, *TAUS, 3.0):
    t0 = time.time()
    f = predict_batch_series_reg(
        fit_pooled_map(sales, Z, X, tau_item=1.0, tau_store=tau), Z_future, X_future, n_draws=1
    )
    overall, items = score(f)
    grid.append(
        {"tau_store": tau, "wrmsse": overall, **dict(zip(items["tier"], items["wrmsse"])), "seconds": time.time() - t0}
    )
tau_grid = pd.DataFrame(grid)
print(tau_grid.round(4).to_string(index=False))

### The same model sampled with NUTS

`ssp_12` found that the posterior mean of `SspReg` did not help while the MAP point estimate did, so the MAP rows above
are the ones to build on. One NUTS run checks that pooling is not an artefact of the point estimate: the model is
`ssp_reg_model` with the pooled prior (`b` a site of its own, shared by the item's stores through the `plate`), one run
per item of 10 stores, 200 warmup draws, 200 draws and 4 chains, as `SspReg` runs. It is scored at the `tau_store` chosen
above; one value only, since a NUTS fit costs minutes.

In [ ]:
def pooled_model(y, Z, X, tau_item, tau_store, sigma_h_prior, sigma_q_prior):
    """``ssp_reg_model`` of one item's stores with ``beta_s = b + delta_s``: ``b`` is one site shared by the stores."""
    n_series, n_states = y.shape[0], Z.shape[1]
    with numpyro.plate("series", n_series):
        sigma_h = numpyro.sample("sigma_h", sigma_h_prior.distribution)
        sigma_q_level = numpyro.sample("sigma_q_level", sigma_q_prior.distribution)
        sigma_q_dow = numpyro.sample("sigma_q_dow", sigma_q_prior.distribution)
    b = numpyro.sample("b", dist.Normal(0.0, tau_item).expand([X.shape[1]]).to_event(1))  # (K,), shared
    with numpyro.plate("series", n_series):
        delta = numpyro.sample("delta", dist.Normal(0.0, tau_store).expand([X.shape[1]]).to_event(1))  # (B, K)
        beta = numpyro.deterministic("beta", b + delta)
        y = y - beta @ X.T
    sigma_q = jnp.column_stack([sigma_q_level, jnp.broadcast_to(sigma_q_dow[:, None], (n_series, n_states - 1))])
    lp, *_ = kalman_filter_1d_batch(
        a0=jnp.zeros(n_states),
        P0=jnp.ones(n_states),
        Z=Z,
        sigma_h=sigma_h,
        sigma_q=sigma_q,
        y=y,
        logp=True,
        chunk_size=None,
    )
    numpyro.factor("log_lik", lp.sum())


def fit_pooled_nuts(tau_store, tau_item=1.0, num_warmup=200, num_samples=200, num_chains=4, seed=0):
    """NUTS on each item's 10 stores, then the posterior-mean forecast of ``predict_batch_series_reg``."""
    forecasts, info = [], []
    for i in range(N_ITEMS):
        rows = slice(i * N_STORES, (i + 1) * N_STORES)
        params_t = transform.fit(sales[rows])
        y = jnp.asarray(transform.forward(sales[rows], params_t))
        mcmc = MCMC(
            NUTS(pooled_model),
            num_warmup=num_warmup,
            num_samples=num_samples,
            num_chains=num_chains,
            chain_method="vectorized",
            progress_bar=False,
        )
        mcmc.run(
            jax.random.PRNGKey(seed),
            y,
            jnp.asarray(Z, jnp.float32),
            jnp.asarray(X, jnp.float32),
            tau_item,
            tau_store,
            SIGMA_H,
            SIGMA_Q,
            extra_fields=("diverging",),
        )
        grouped = mcmc.get_samples(group_by_chain=True)
        samples = {
            k: np.asarray(v).reshape(-1, *v.shape[2:])
            for k, v in grouped.items()
            if k in ("sigma_h", "sigma_q_level", "sigma_q_dow", "beta")
        }
        fit = {
            "samples": samples,
            "y": np.asarray(y),
            "transform_params": params_t,
            "transform": transform,
            "Z": jnp.asarray(Z, jnp.float32),
            "X": np.asarray(X, np.float32),
        }
        forecasts.append(predict_batch_series_reg(fit, Z_future, X_future, n_draws=100))
        info.append(
            {
                "divergences": int(np.asarray(mcmc.get_extra_fields()["diverging"]).sum()),
                "n_eff_min_b": float(np.asarray(effective_sample_size(np.asarray(grouped["b"]))).min()),
            }
        )
    return np.vstack(forecasts), info


t0 = time.time()
f_nuts, nuts_info = fit_pooled_nuts(tau_best)
print(record(f"beta: partial pooling (tau={tau_best:g}), NUTS", f_nuts, time.time() - t0, group="coefficients"))
print("divergences and smallest effective sample size of the item coefficients, by item:", nuts_info)

## A joint filter per item

The weekly and level variants share filter states across an item's stores, so the 10 stores are filtered together:
10 observations a day, a state vector of the 10 store levels (and, where used, six weekday offsets shared by the
stores), and the full covariance between the states kept. This is the "coefficients as filter states" route of the issue,
which the batch fitter does not support: `kalman_filter_1d_batch` runs independent series through a diagonal-covariance
filter. `joint_filter` below is a plain linear Gaussian filter in Joseph form, scanned over the days; the three items
are `vmap`ped. Its check against the batch filter comes first, with independent levels and no shared weekday.

In [ ]:
LOG_2PI = float(np.log(2 * np.pi))


def joint_filter(y, Zd, Q, sigma_h, with_dow):
    """Kalman filter of one item: ``n_lvl`` store levels, plus six weekday offsets shared by the stores if ``with_dow``.

    Parameters
    ----------
    y : (T, n_lvl) transformed sales of the item's stores.
    Zd : (T, 6) weekday dummies.
    Q : (n, n) covariance of the state innovations; ``n = n_lvl`` or ``n_lvl + 6``. Random-walk states.
    sigma_h : (n_lvl,) observation noise of each store.

    Returns
    -------
    log_lik, a_T, P_T : the log-likelihood and the terminal one-step-ahead state mean and covariance.
    """
    n, n_lvl = Q.shape[0], y.shape[1]
    R, eye = jnp.diag(sigma_h**2), jnp.eye(n)

    def step(carry, inputs):
        a, P, lp = carry
        y_t, d_t = inputs
        Z_t = (
            jnp.concatenate([jnp.eye(n_lvl), jnp.broadcast_to(d_t, (n_lvl, 6))], axis=1) if with_dow else jnp.eye(n_lvl)
        )
        v = y_t - Z_t @ a
        chol = jnp.linalg.cholesky(Z_t @ P @ Z_t.T + R)
        w = jsp.linalg.solve_triangular(chol, v, lower=True)
        lp = lp - 0.5 * (2 * jnp.log(jnp.diagonal(chol)).sum() + w @ w + n_lvl * LOG_2PI)
        gain = jsp.linalg.cho_solve((chol, True), Z_t @ P).T
        a_f = a + gain @ v
        IKZ = eye - gain @ Z_t
        P_f = IKZ @ P @ IKZ.T + gain @ R @ gain.T
        return (a_f, 0.5 * (P_f + P_f.T) + Q, lp), None

    (a, P, lp), _ = jax.lax.scan(step, (jnp.zeros(n), jnp.eye(n), 0.0), (y, Zd))
    return lp, a, P


# Check: independent levels and no weekday state, against the batch filter of the 10 stores of the first item.
yt = jnp.asarray(transform.forward(sales[:N_STORES], transform.fit(sales[:N_STORES])))
sigma_h_c, sigma_q_c = jnp.full(N_STORES, 0.5), jnp.full(N_STORES, 0.05)
lp_batch, *_ = kalman_filter_1d_batch(
    a0=jnp.zeros(1),
    P0=jnp.ones(1),
    Z=jnp.ones((n_fit, 1)),
    sigma_h=sigma_h_c,
    sigma_q=sigma_q_c[:, None],
    y=yt,
    logp=True,
    chunk_size=None,
)
lp_joint, *_ = joint_filter(yt.T, jnp.asarray(Z[:, 1:]), jnp.diag(sigma_q_c**2), sigma_h_c, False)
print(f"log-likelihood, batch filter {float(lp_batch.sum()):.2f}, joint filter {float(lp_joint):.2f}")

### A MAP fitter for the joint filter

`fit_joint_map` fits the three items at once (a `vmap` of `joint_filter`), by Adam on the same posterior as above, and
forecasts from the terminal filtered state, with the horizon-flat variance of `predict_batch_series_opt` (the terminal
filtered variance plus `sigma_h^2`). Its options:

* `dow="static"`: no weekday state; the six weekday offsets are static coefficients, which can be pooled like `beta`
  (`dow_pool = (tau_item, tau_store)`, `None` leaving a part out). `dow="shared"`: six drifting offsets shared by the
  item's 10 stores as filter states, optionally with a static store deviation (`dow_pool = (None, tau_store)`).
* `cov_pool = (tau_item, tau_store)`: the pooling of the 73 covariate coefficients, as before.
* `level`: the covariance of the 10 store-level innovations. `"indep"`: diagonal. `"equicorr"`: `diag(s^2) + c^2 11'`, one
  shock shared equally by the stores. `"factor"`: `diag(s^2) + l l'`, one common item level that each store scales by
  `l_s` (a one-factor SUTSE). `"full"`: `diag(s) C diag(s)` with `C` a full correlation matrix (a row-normalised
  Cholesky factor).

In [ ]:
HALF_NORMAL_Q = dist.HalfNormal(0.05)  # the prior of the drifts, as SIGMA_Q


def _corr_factor(raw):
    """Row-normalised lower-triangular factor ``L`` of a correlation matrix ``L L'`` from unconstrained entries."""
    L = jnp.tril(raw, -1) + jnp.diag(_softplus(jnp.diagonal(raw)))
    return L / jnp.linalg.norm(L, axis=1, keepdims=True)


def fit_joint_map(
    sales,
    Z,
    X,
    Z_future,
    X_future,
    level="indep",
    dow="static",
    cov_pool=(None, 1.0),
    dow_pool=(None, 1.0),
    n_iter=300,
    lr=3e-2,
):
    """MAP fit of each item as one multivariate model over its stores; returns the 28-day forecasts and some info."""
    params_t = transform.fit(sales)
    y = jnp.asarray(transform.forward(sales, params_t))
    n_days = y.shape[1]
    y_item = y.reshape(N_ITEMS, N_STORES, n_days).transpose(0, 2, 1)  # (item, day, store)
    Zd, Zd_future = jnp.asarray(Z[:, 1:], jnp.float32), np.asarray(Z_future[:, 1:], np.float32)
    Xj = jnp.asarray(X, jnp.float32)
    with_dow = dow == "shared"
    if dow == "static" and dow_pool == (None, None):
        raise ValueError("a static weekday pattern needs a coefficient: set dow_pool")

    def coefficients(p, name, pool, width):
        c = jnp.zeros((N_ITEMS, N_STORES, width))
        if pool[0] is not None:
            c = c + p[name + "_item"][:, None, :]
        if pool[1] is not None:
            c = c + p[name + "_store"]
        return c

    def innovations(p):
        sigma_h, s = _softplus(p["raw_h"]), _softplus(p["raw_q"])  # (item, store)
        if level == "indep":
            sigma = jax.vmap(jnp.diag)(s**2)
        elif level == "equicorr":
            sigma = jax.vmap(jnp.diag)(s**2) + (_softplus(p["raw_c"]) ** 2)[:, None, None]
        elif level == "factor":
            lam = _softplus(p["raw_l"])
            sigma = jax.vmap(jnp.diag)(s**2) + lam[:, :, None] * lam[:, None, :]
        else:  # "full"
            L = jax.vmap(_corr_factor)(p["raw_w"])
            sigma = s[:, :, None] * (L @ L.transpose(0, 2, 1)) * s[:, None, :]
        if not with_dow:
            return sigma_h, sigma, None
        sigma_d = _softplus(p["raw_d"])
        Q = jax.vmap(lambda S_, d: jsp.linalg.block_diag(S_, d**2 * jnp.eye(6)))(sigma, sigma_d)
        return sigma_h, Q, sigma_d

    def residual(p):
        cov, wk = coefficients(p, "cov", cov_pool, K), coefficients(p, "dow", dow_pool, 6)
        static = jnp.einsum("isk,tk->ist", cov, Xj) + jnp.einsum("isk,tk->ist", wk, Zd)
        return y_item - static.transpose(0, 2, 1), cov, wk

    def neg_log_post(p):
        sigma_h, Q, sigma_d = innovations(p)
        r, _, _ = residual(p)
        lp = jax.vmap(lambda y_, Q_, h_: joint_filter(y_, Zd, Q_, h_, with_dow)[0])(r, Q, sigma_h)
        prior = dist.HalfNormal(1.0).log_prob(sigma_h).sum() + HALF_NORMAL_Q.log_prob(_softplus(p["raw_q"])).sum()
        if with_dow:
            prior += HALF_NORMAL_Q.log_prob(sigma_d).sum()
        for key in ("raw_c", "raw_l"):  # the common shock of "equicorr" and the loadings of "factor"
            if key in p:
                prior += HALF_NORMAL_Q.log_prob(_softplus(p[key])).sum()
        if "raw_w" in p:
            prior += dist.Normal(0.0, 1.0).log_prob(p["raw_w"]).sum()
        for name, pool in (("cov", cov_pool), ("dow", dow_pool)):
            if pool[0] is not None:
                prior += dist.Normal(0.0, pool[0]).log_prob(p[name + "_item"]).sum()
            if pool[1] is not None:
                prior += dist.Normal(0.0, pool[1]).log_prob(p[name + "_store"]).sum()
        return -(lp.sum() + prior)

    p = {
        "raw_h": jnp.full((N_ITEMS, N_STORES), _softplus_inv(SIGMA_H.init)),
        "raw_q": jnp.full((N_ITEMS, N_STORES), _softplus_inv(SIGMA_Q.init)),
    }
    if level == "equicorr":
        p["raw_c"] = jnp.full((N_ITEMS,), _softplus_inv(0.02))
    if level == "factor":
        p["raw_l"] = jnp.full((N_ITEMS, N_STORES), _softplus_inv(0.02))
    if level == "full":
        p["raw_w"] = jnp.tile(jnp.eye(N_STORES) * _softplus_inv(1.0), (N_ITEMS, 1, 1))
    if with_dow:
        p["raw_d"] = jnp.full((N_ITEMS,), _softplus_inv(SIGMA_Q.init))
    for name, pool, width in (("cov", cov_pool, K), ("dow", dow_pool, 6)):
        if pool[0] is not None:
            p[name + "_item"] = jnp.zeros((N_ITEMS, width))
        if pool[1] is not None:
            p[name + "_store"] = jnp.zeros((N_ITEMS, N_STORES, width))

    opt = optax.adam(lr)
    state = opt.init(p)

    @jax.jit
    def step(p, state):
        loss, grads = jax.value_and_grad(neg_log_post)(p)
        updates, state = opt.update(grads, state)
        return optax.apply_updates(p, updates), state, loss

    for _ in range(n_iter):
        p, state, loss = step(p, state)

    # The forecast: the terminal state, the static terms, and the transform's inverse with the terminal variance.
    sigma_h, Q, _ = innovations(p)
    r, cov, wk = residual(p)
    _, a_T, P_T = jax.vmap(lambda y_, Q_, h_: joint_filter(y_, Zd, Q_, h_, with_dow))(r, Q, sigma_h)
    a_T, P_T, sigma_h = np.asarray(a_T), np.asarray(P_T), np.asarray(sigma_h)
    horizon = Zd_future.shape[0]
    mu = np.zeros((N_ITEMS, N_STORES, horizon))
    var = np.zeros_like(mu)
    for i in range(N_ITEMS):
        for h in range(horizon):
            Z_h = np.eye(N_STORES)
            if with_dow:
                Z_h = np.concatenate([Z_h, np.tile(Zd_future[h], (N_STORES, 1))], axis=1)
            mu[i, :, h] = Z_h @ a_T[i]
            var[i, :, h] = np.einsum("sn,nm,sm->s", Z_h, P_T[i], Z_h) + sigma_h[i] ** 2
    mu += np.einsum("isk,hk->ish", np.asarray(cov), np.asarray(X_future, np.float32))
    mu += np.einsum("isk,hk->ish", np.asarray(wk), Zd_future)
    forecast = transform.inverse(mu.reshape(-1, horizon), var.reshape(-1, horizon), params_t)
    return np.asarray(forecast), {"loss": float(loss), "Q": np.asarray(Q), "params": p}

In [ ]:
def joint_row(name, group, **kw):
    """Fit the joint model, score it and keep the row."""
    t0 = time.time()
    f, info = fit_joint_map(sales, Z, X, Z_future, X_future, **kw)
    print(record(name, f, time.time() - t0, group=group))
    return f, info


def holdout_joint(**kw):
    """The mean holdout RMSSE of a joint fit over the three rolling folds."""
    errs = []
    for cut in CUTS:
        pred, _ = fit_joint_map(sales[:, :cut], Z[:cut], X[:cut], Z[cut : cut + HOLDOUT], X[cut : cut + HOLDOUT], **kw)
        errs.append(np.mean(holdout_rmsse(pred, sales[:, cut : cut + HOLDOUT], sales[:, :cut])))
    return float(np.mean(errs))

In [ ]:
# The batch filter keeps only the diagonal of the state covariance, which is exact for one state and approximate for
# seven: the log-likelihood of one series (the first store of the high item) with a level and six weekday offsets.
y1 = jnp.asarray(transform.forward(sales[:1], transform.fit(sales[:1])))
lp_batch7, *_ = kalman_filter_1d_batch(
    a0=jnp.zeros(7),
    P0=jnp.ones(7),
    Z=jnp.asarray(Z, jnp.float32),
    sigma_h=jnp.array([0.5]),
    sigma_q=jnp.array([[0.05] + [0.01] * 6]),
    y=y1,
    logp=True,
    chunk_size=None,
)
lp_joint7, *_ = joint_filter(
    y1.T, jnp.asarray(Z[:, 1:]), jnp.diag(jnp.array([0.05**2] + [0.01**2] * 6)), jnp.array([0.5]), True
)
print(
    f"log-likelihood with 7 states, diagonal batch filter {float(lp_batch7.sum()):.2f}, exact joint filter {float(lp_joint7):.2f}"
)

## Sharing the weekly pattern

The #118 default gives every store its own weekday offsets, drifting as a random walk. Here the weekly pattern is moved
into the joint filter's side:

* **static, own:** the six offsets are static coefficients of the store, `N(0, 1)`. A store's level still drifts. Compared
  with the default, this says what a static pattern costs (and with the covariates, in the same fit as `beta`).
* **static, full pooling:** one static pattern per item, shared by the 10 stores.
* **static, partial pooling:** the item's pattern plus a store deviation with scale `tau`, chosen on the rolling folds.
* **drifting, shared:** six drifting offsets per item, filter states shared by the 10 stores, with or without a static
  store deviation.

All use independent store levels and `beta` under `N(0, 1)` per store (no pooling), so the weekly pattern is the only
change from row to row. The first row has no covariates at all.

In [ ]:
base = dict(level="indep", dow="static", cov_pool=(None, 1.0))
no_cov = dict(level="indep", dow="static", cov_pool=(None, None))
joint_row("weekly: static own, no covariates", "weekly pattern", dow_pool=(None, 1.0), **no_cov)
f_w_own, _ = joint_row("weekly: static own pattern", "weekly pattern", dow_pool=(None, 1.0), **base)
joint_row("weekly: static full pooling, no covariates", "weekly pattern", dow_pool=(1.0, None), **no_cov)
f_w_full, _ = joint_row("weekly: static full pooling", "weekly pattern", dow_pool=(1.0, None), **base)

DOW_TAUS = (0.03, 0.1, 0.3, 1.0)
t0 = time.time()
dow_holdout = {tau: holdout_joint(dow_pool=(1.0, tau), **base) for tau in DOW_TAUS}
tau_dow = min(dow_holdout, key=dow_holdout.get)
print(
    "holdout RMSSE by tau:",
    {k: round(v, 4) for k, v in dow_holdout.items()},
    "-> tau =",
    tau_dow,
    f"({time.time() - t0:.0f} s to choose)",
)
f_w_part, _ = joint_row(
    f"weekly: static partial pooling (tau={tau_dow:g}, holdout choice)",
    "weekly pattern",
    dow_pool=(1.0, tau_dow),
    **base,
)

_, shared_info = joint_row(
    "weekly: drifting shared, no deviation",
    "weekly pattern",
    level="indep",
    dow="shared",
    dow_pool=(None, None),
    cov_pool=(None, 1.0),
)
print(
    "fitted drift of the shared weekday offsets, by item:",
    np.asarray(_softplus(shared_info["params"]["raw_d"])).round(4),
)
joint_row(
    "weekly: drifting shared + store deviation (tau=0.3)",
    "weekly pattern",
    level="indep",
    dow="shared",
    dow_pool=(None, 0.3),
    cov_pool=(None, 1.0),
)

In [ ]:
# Sensitivity of the pre-validation score to the weekly pooling scale. As above, this reads the scored origin: it shows
# how flat the choice is, it is not a choice. tau -> 0 is full pooling and a large tau is no pooling.
grid = []
for tau in DOW_TAUS:
    f, _ = fit_joint_map(sales, Z, X, Z_future, X_future, dow_pool=(1.0, tau), **base)
    overall, items = score(f)
    grid.append({"tau_store": tau, "wrmsse": overall, **dict(zip(items["tier"], items["wrmsse"]))})
dow_grid = pd.DataFrame(grid)
print(dow_grid.round(4).to_string(index=False))

## Combining the pooled weekly pattern and the pooled coefficients

The two poolings above are separate changes. The joint filter can take both at once, with independent levels.

In [ ]:
f_comb_full, _ = joint_row(
    "combined: weekly full pooling + beta partial pooling",
    "combined",
    level="indep",
    dow="static",
    dow_pool=(1.0, None),
    cov_pool=(1.0, tau_best),
)
f_comb_both, _ = joint_row(
    "combined: weekly full pooling + beta full pooling",
    "combined",
    level="indep",
    dow="static",
    dow_pool=(1.0, None),
    cov_pool=(1.0, None),
)

With independent levels and a static weekly pattern, nothing is shared through the filter: given the coefficients, each
store's filter is independent, and only the prior of the coefficients couples the stores. So the weekly variants need no
joint filter. The same fit runs on the independent level-only filters of `fit_pooled_map`, with the six weekday dummies as
six more static columns beside the covariates (here both pooled in full), and should reproduce the joint row.

In [ ]:
X_week, X_week_future = np.concatenate([Z[:, 1:], X], axis=1), np.concatenate([Z_future[:, 1:], X_future], axis=1)
level_only, level_only_future = np.ones((n_fit, 1), np.float32), np.ones((Z_future.shape[0], 1), np.float32)
t0 = time.time()
res = fit_pooled_map(sales, level_only, X_week, tau_item=1.0)  # weekday and covariate coefficients, item part only
f_indep_route = predict_batch_series_reg(res, level_only_future, X_week_future, n_draws=1)
print(record("combined: the same, independent level-only filters", f_indep_route, time.time() - t0, group="combined"))
print("largest difference to the joint row's forecasts:", float(np.abs(f_indep_route - f_comb_both).max()))

## Correlated store levels

The level innovations of an item's 10 stores are correlated in the joint filter (SUTSE): `equicorr` (one shock shared
equally), `factor` (one common item level that each store scales) and `full` (a full correlation matrix). Weekly pattern
and covariates are the static, per-store ones of the first weekly row, so the level covariance is the only change from it.

In [ ]:
level_info = {}
for level in ("indep", "equicorr", "factor", "full"):
    _, level_info[level] = joint_row(
        f"levels: {level}", "store levels", level=level, dow="static", dow_pool=(None, 1.0), cov_pool=(None, 1.0)
    )
# What the fitted level noise looks like: the innovation sd of a store, and the mean correlation between stores.
for level, info in level_info.items():
    sd = np.sqrt(np.diagonal(info["Q"], axis1=1, axis2=2))
    corr = info["Q"] / sd[:, :, None] / sd[:, None, :]
    off = ~np.eye(N_STORES, dtype=bool)
    print(
        f"{level:9s} innovation sd of a store, by item {sd.mean(axis=1).round(3)}, mean correlation {[round(float(c[off].mean()), 2) for c in corr]}"
    )

### The other two origins

One 28-day window on 30 series is a small sample. The MAP fits are cheap, so the main variants are also scored at the
validation and evaluation origins (the scores are read once, not used to choose anything: `tau` is the pre-validation
choice). The NUTS fit is not repeated.

In [ ]:
VARIANTS = {
    "weekly: static own": dict(kind="joint", dow_pool=(None, 1.0), cov_pool=(None, 1.0)),
    "weekly: static full pooling": dict(kind="joint", dow_pool=(1.0, None), cov_pool=(None, 1.0)),
    f"weekly: static partial pooling (tau={tau_dow:g})": dict(
        kind="joint", dow_pool=(1.0, tau_dow), cov_pool=(None, 1.0)
    ),
    "combined: weekly full + beta partial": dict(kind="joint", dow_pool=(1.0, None), cov_pool=(1.0, tau_best)),
    "levels: equicorr": dict(kind="joint", level="equicorr", dow_pool=(None, 1.0), cov_pool=(None, 1.0)),
    "beta: no pooling (ridge)": dict(kind="indep", tau_store=1.0),
    "beta: full pooling": dict(kind="indep", tau_item=1.0),
    f"beta: partial pooling (tau={tau_best:g})": dict(kind="indep", tau_item=1.0, tau_store=tau_best),
}
origin_rows = []
for window in ("prevalidation", "validation", "evaluation"):
    fm, fcm = window_masks(ds["split"].values, window)
    if not np.isfinite(ds["sales"].values[series_idx][:, fcm]).all():
        print(f"{window}: no actuals in the dataset, skipped")
        continue
    sales_w = ds["sales"].values[series_idx][:, fm]
    base_f, _ = MODELS[DEFAULT_MODEL].fit_forecast(ds, series_idx, fm, fcm)
    origin_rows.append({"origin": window, "approach": "#118 default", "wrmsse": score(base_f, window)[0]})
    for label, spec in VARIANTS.items():
        spec = dict(spec)
        if spec.pop("kind") == "joint":
            f, _ = fit_joint_map(
                sales_w, Z_all[fm], X_all[fm], Z_all[fcm], X_all[fcm], **{"level": "indep", "dow": "static", **spec}
            )
        else:
            f = predict_batch_series_reg(
                fit_pooled_map(sales_w, Z_all[fm], X_all[fm], **spec), Z_all[fcm], X_all[fcm], n_draws=1
            )
        origin_rows.append({"origin": window, "approach": label, "wrmsse": score(f, window)[0]})
origins = pd.DataFrame(origin_rows).pivot(index="approach", columns="origin", values="wrmsse")
print(origins[[c for c in ("prevalidation", "validation", "evaluation") if c in origins]].round(4).to_string())

## All variants, side by side

In [ ]:
results = pd.DataFrame(RESULTS).set_index("approach")
results["vs_default"] = results["wrmsse"] - results.loc["#118 default (MAP, no covariates)", "wrmsse"]
show = ["group", "wrmsse", "vs_default", *[c for c in results.columns if c.startswith("wrmsse_")], "seconds"]
print(f"Pre-validation WRMSSE of the 3-item x 10-store subset (lower is better; {K} covariates where used)")
print(results[show].round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2.2, 1]})
colors = {g: f"C{i}" for i, g in enumerate(results["group"].unique())}
order = results["wrmsse"].sort_values().index
axes[0].barh(range(len(order)), results.loc[order, "wrmsse"], color=[colors[g] for g in results.loc[order, "group"]])
axes[0].set_yticks(range(len(order)), order, fontsize=8)
axes[0].axvline(results.loc["#118 default (MAP, no covariates)", "wrmsse"], color="k", ls="--", lw=1)
axes[0].set_xlim(0.5, min(1.0, results["wrmsse"].max() * 1.02))
axes[0].invert_yaxis()
axes[0].set_xlabel("pre-validation WRMSSE (dashed: #118 default)")
axes[0].legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()], colors, fontsize=8, loc="upper right")
axes[1].plot(tau_grid["tau_store"], tau_grid["wrmsse"], "o-", label="beta pooling (independent filters)")
axes[1].plot(dow_grid["tau_store"], dow_grid["wrmsse"], "s-", label="weekly pattern pooling (joint filter)")
axes[1].axhline(results.loc["#118 default (MAP, no covariates)", "wrmsse"], color="k", ls="--", lw=1)
axes[1].set_xscale("log")
axes[1].set_xlabel("tau of the store deviation")
axes[1].set_ylabel("pre-validation WRMSSE")
axes[1].set_title("tau -> 0: full pooling; large: none", fontsize=9)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Findings

Numbers are from one run of this notebook on the 30-series subset (pre-validation unless said otherwise; WRMSSE, lower is
better; "vs default" is the change from the #118 default, 0.5385). `high`, `mid` and `low` are the item-level scores of
`FOODS_3_586`, `HOUSEHOLD_1_069` and `HOBBIES_1_197`; the high item holds 96% of the dollar weight.

| variant | WRMSSE | vs default | high / mid / low | fit time |
|---|---|---|---|---|
| #118 default (own drifting weekly pattern, no covariates) | 0.5385 | | 0.534 / 0.599 / 0.719 | 1.2 s |
| reconciled forecasts, OLS / WLS / MinT-shrink (bottom-up is the default) | 0.5939 / 0.5678 / 0.5516 | +0.055 / +0.029 / +0.013 | MinT: 0.548 / 0.589 / 0.718 | 1.9 s with the base fits (MinT needs 6 more for its errors) |
| `beta` no pooling (ridge, `tau` = 1) | 0.5227 | -0.0158 | 0.516 / 0.622 / 0.722 | 1.6 s |
| `beta` full pooling | 0.5617 | +0.0232 | 0.557 / 0.609 / 0.719 | 1.6 s |
| `beta` partial pooling, `tau` = 0.03 (holdout choice) | 0.5568 | +0.0183 | 0.552 / 0.610 / 0.720 | 1.6 s |
| `beta` partial pooling, `tau` = 0.03, NUTS | 0.5721 | +0.0336 | 0.568 / 0.611 / 0.719 | 351 s |
| static weekly pattern, own, no covariates | 0.5441 | +0.0056 | 0.539 / 0.624 / 0.712 | 19 s |
| static weekly pattern, full pooling, no covariates | 0.5411 | +0.0026 | 0.536 / 0.611 / 0.712 | 19 s |
| static weekly pattern, own, with covariates | 0.5283 | -0.0102 | 0.522 / 0.639 / 0.717 | 19 s |
| **static weekly pattern, full pooling, with covariates** | **0.5169** | **-0.0216** | 0.511 / 0.622 / 0.717 | 19 s |
| static weekly pattern, partial pooling, `tau` = 0.1 (holdout choice) | 0.5260 | -0.0125 | 0.520 / 0.632 / 0.716 | 19 s |
| shared drifting weekly pattern, no deviation / with deviation | 0.6298 / 0.6576 | +0.091 / +0.119 | 0.628 / 0.607 / 0.719 | 21 s |
| weekly full pooling + `beta` partial pooling / full pooling | 0.5602 / 0.5654 | +0.022 / +0.027 | 0.556 / 0.616 / 0.714 | 19 s |
| correlated levels: equicorrelated / one-factor / full | 0.8703 / 0.8583 / 0.8674 | +0.33 / +0.32 / +0.33 | 0.876 / 0.630 / 0.717 | 19 s |

Across origins (pre-validation / validation / evaluation; the default is 0.5385 / 0.5155 / 0.5938):

| variant | pre-validation | validation | evaluation |
|---|---|---|---|
| `beta` no pooling | 0.5227 | 0.5088 | 0.5724 |
| `beta` full pooling | 0.5617 | 0.4997 | 0.5489 |
| `beta` partial pooling, `tau` = 0.03 | 0.5568 | 0.4993 | 0.5500 |
| static weekly pattern, own | 0.5283 | 0.5297 | 0.5692 |
| static weekly pattern, full pooling | 0.5169 | 0.5227 | 0.5674 |
| static weekly pattern, partial pooling, `tau` = 0.1 | 0.5260 | 0.5285 | 0.5678 |
| weekly full pooling + `beta` partial pooling | 0.5602 | 0.5132 | 0.5392 |
| correlated levels, equicorrelated | 0.8703 | 0.6570 | 0.6340 |

1. **Pooling the weekly pattern is the variant that helps.** One static weekly pattern per item beats the store's own
   static pattern at all three origins (0.5169 against 0.5283, 0.5227 against 0.5297, 0.5674 against 0.5692), with and
   without covariates (0.5411 against 0.5441), and it is the best row at pre-validation, 0.0216 under the default. The
   gain is in the high item (0.534 → 0.511). Two cautions. A static pattern alone costs something against the default's
   drifting one (0.5441 against 0.5385 with no covariates), and at the validation origin every row with a static pattern and per-store coefficients is
   behind the default (0.5227-0.5297 against 0.5155), so what pooling buys is the gap between the static rows, not a win at
   every origin. The scored-origin sensitivity grid is monotone in the pooling (0.5184 at `tau` = 0.03, 0.5286 at 1), and
   the holdout, which is flat (0.631-0.632 across the grid), picks `tau` = 0.1.
2. **Pooling the covariate coefficients does not help at pre-validation, and its sign flips with the origin.** Full pooling
   is 0.039 behind no pooling at pre-validation (0.5617 against 0.5227), but 0.009 and 0.024 ahead at the validation and
   evaluation origins (0.4997 and 0.5489 against 0.5088 and 0.5724). The high item wants store-specific coefficients at
   pre-validation (0.557 against 0.516); the mid item gets better with pooling (0.622 → 0.609, which brings it back toward
   the default's 0.599). The holdout picks `tau` = 0.03, almost full pooling, because its RMSSE is an unweighted mean over
   30 series, and 20 of them are the mid and low items, whereas the WRMSSE is 96% the high item. The best `tau` on the scored
   origin (0.3, 0.5211) is only 0.0016 under no pooling, which is no gain. Over the three origins the unpooled, fully
   pooled and partially pooled `beta` rows average 0.5346, 0.5368 and 0.5354: the differences are noise on 3 items.
3. **NUTS does not help the pooled model either.** At `tau` = 0.03 it scores 0.5721 against the MAP fit's 0.5568, takes 351 s
   against 1.6 s, and the item coefficients mix badly for the mid and low items (smallest effective sample sizes 31 and 22
   in 800 draws), as `SspReg` did in `ssp_12`.
4. **A shared drifting weekly pattern is much worse.** 0.6298, and 0.6576 with a store deviation, against 0.5283 for a static
   pattern per store; the high item goes from 0.522 to 0.628-0.656. The exact joint filter lets the shared offsets drift
   (the fitted drift of the high item's offsets is 0.082 a day, against 0.003 for the other two items), and the forecast
   then starts from wherever the last weeks left the pattern. The batch filter keeps only the diagonal of the state
   covariance, which is exact for one state and not for seven (the check above: -2147.2 against the exact -2151.8 on one
   series), so the default's own filter is not the joint filter's model with the stores unpooled.
5. **Correlated store levels are much worse, at every origin.** All three SUTSE variants score 0.86-0.87 against 0.5283 for
   independent levels in the same fit, and the equicorrelated one is also worse at validation and evaluation (0.657 and
   0.634 against 0.530 and 0.569). The high item's fitted level noise is large and common: 0.18-0.21 a day per store against
   0.079, with a mean correlation of 0.79-0.89 between stores. A shock that all 10 stores share is what the weekly and
   calendar effects of a static pattern leave over, and a persistent level absorbs it, so the forecast starts from the last
   day's shock. The mid item gains a little (0.639 → 0.628-0.633), the low item does not move.
6. **Reconciling independent forecasts is not worth it here.** Bottom-up is the default's score. The item totals' own
   forecasts are worse than the sum of the stores' (the ratio is 1.03, 0.92 and 0.97 by item), so combining them moves the
   high item from 0.534 to 0.548 (MinT-shrink) - 0.592 (OLS). It helps only the mid item (0.599 → 0.583-0.589), and the shrinkage
   intensities (0.14, 0.47, 0.40 by item) come from 84 error vectors that are not independent.
7. **The joint filter is not needed for the variant that works.** With independent levels and a static weekly pattern the
   stores are independent given the coefficients, so the weekly pooling runs on the batch filter with a level-only state
   and the weekday dummies as static columns: the cell above reproduces the joint row (0.5654) in 2 s instead of 19 s. The
   joint filter is only needed for the shared drifting and correlated-level variants, which lost.

Limits of this evidence: 30 series from 3 items, one 28-day window per origin, MAP point estimates (one NUTS fit), one seed.
The `beta` pooling results change sign with the origin, so they should be read as unresolved, not as a ranking. Differences of
0.005 or less should not be read as a ranking either. The default itself has 0.0004 of slack between two implementations of
the same model (`ssp_12`).

### What each variant needs

| variant | needs | missing |
|---|---|---|
| Weekly pattern and `beta` pooled across an item's stores, independent filters (built here as `fit_pooled_map`) | a MAP fitter for `y - X beta` with a level-only filter and the weekday dummies as static columns, in `src/bunobee/models/m5/m5_ssp_optim.py` (as in the `ssp_12` recommendation); an `item` index of the batch, so that an item's 10 stores are fitted together; `predict_batch_series_reg` for the forecast | the item index in `fit_forecast` (the entries get `series_idx` only), and `tau_item` and `tau_store` fields |
| The same, sampled (built here as `pooled_model`) | `MCMC`, `NUTS`, `plate`; one extra `b` site beside `ssp_reg_model`'s `beta` | nothing missing; too slow and no better |
| Joint filter per item (built here as `joint_filter`) | `jax.lax.scan`, `jax.scipy.linalg.cho_solve` | not in `src/bunobee/models/ssp/` (only the diagonal batch filter); needed only by the variants that lost |
| Correlated levels, shared drifting weekly pattern (built here) | the joint filter | no use for them, on this evidence |
| Reconciliation (built here as `combine`) | a forecast of each item total and the 11 × 10 summing matrix | no model code; no gain |

### Recommendation

**Register one entry: an `SspReg`-style MAP fit with a level-only filter, a static weekly pattern pooled across the item's
stores, and the 73 covariates per store.** It is the best row at pre-validation (0.5169, 0.0216 under the default), it beats its
own unpooled twin at all three origins, and it fits 30 series in 2 s on the batch filter. Give it two settings: the weekly
pooling scale (`tau_store` = 0 for full pooling is the default, 0.1 the holdout choice) and the covariate prior (`tau` = 1, per
store). Its two honest weaknesses are the mid item (0.622 against the default's 0.599, as in `ssp_12`) and the validation
origin, where it is behind the default.

**A second setting, not a second entry: partial pooling of `beta` (`tau_store` about 0.03) on top of it.** It is behind at
pre-validation (0.5602 against 0.5169) and the best at evaluation (0.5392 against 0.5674 for the same weekly pooling with a per-store `beta`), so the
re-plan's rule (pre-validation decides) rejects it for now. Whether it is worth keeping should be settled on the
3,000-series sample, which has many more items than 3, not on this subset.

**Do not register:** a shared drifting weekly pattern, correlated store levels (any of the three), reconciliation, or the NUTS
route. They lose here by 0.01-0.33, and the first two need a joint filter that the project does not have. If correlated
levels are tried again, do it with a weekly pattern that is not static, so that common calendar shocks do not end up in the
level.